# Soccer Forecast Agent — Final Submission

**Multi-Agent AI System for Premier League Match Forecasting and Alert Generation**

---

This notebook demonstrates the complete end-to-end *Soccer Forecast Agent*: a production-grade
multi-agent AI system that monitors Premier League fixtures, computes probabilistic forecasts
using a Dixon-Coles statistical model enhanced with recency-weighted form, gathers qualitative
news evidence via a ReAct research loop, and fires styled HTML email alerts when a statistically
meaningful edge is detected over market-implied probabilities.

**Stack:** Python 3.11 · LangGraph · OpenAI API · SQLite · ChromaDB · SentenceTransformers · The Odds API · Tavily Search

> **Note — API keys are required for the live pipeline cells.**
> Cells that call external APIs are marked with `# [LIVE]`.
> The notebook is written to degrade gracefully: if live APIs or quotas are unavailable,
> the explanatory sections still run and the report falls back to cached/local results where possible.


---

## System Architecture

The system follows a supervisor-based multi-agent pattern with four specialised agents orchestrated by a
[LangGraph](https://github.com/langchain-ai/langgraph) `StateGraph`. All agents depend only on
abstract `Protocol` interfaces — no concrete SDK import appears inside any agent class
(**Dependency Inversion Principle**).

```
┌───────────────────────────────────────────────────────────────┐
│                      SupervisorAgent                          │
│   LangGraph StateGraph — routes shared state between agents   │
└───────────┬──────────────────────────────────┬────────────────┘
            │                                  │
   ┌────────▼────────┐              ┌──────────▼─────────┐
   │ StatsMarketAgent│              │ NewsContextAgent    │
   │                 │              │                     │
   │ Fetch fixtures  │              │ ReAct loop:         │
   │ Fetch odds      │              │  · seed from Chroma │
   │ Dixon-Coles     │              │  · web search (Tav) │
   │ baseline        │              │  · extract evidence │
   └────────┬────────┘              └──────────┬──────────┘
            │                                  │
            └─────────────┬────────────────────┘
                          │
               ┌──────────▼──────────┐
               │ SynthesisAlertAgent │
               │                     │
               │ LLM adjudication    │
               │ Log-odds adjustment │
               │ AlertGuard checks   │
               │ HTML email dispatch │
               └─────────────────────┘
```

| Agent | Responsibility | SOLID principle highlighted |
|---|---|---|
| `SupervisorAgent` | Orchestrates the match-loop LangGraph workflow | SRP |
| `StatsMarketAgent` | Fetches fixtures + odds, runs Dixon-Coles | OCP (strategy injection) |
| `NewsContextAgent` | ReAct think-act-observe research loop | DIP (LLMProvider protocol) |
| `SynthesisAlertAgent` | LLM synthesis, edge calculation, alert dispatch | ISP (narrow protocols) |

**Persistence:** SQLite (structured — fixtures, forecasts, evidence) + ChromaDB (vector — article chunks)


### What The System Is Trying To Do

At a high level, the project answers one practical question:

> **Given upcoming Premier League matches, can we combine a statistical prior, live qualitative news, and market odds to decide whether an alert-worthy betting edge exists?**

The system is intentionally split into layers so the reasoning is auditable:

1. **Statistical prior** — a Dixon-Coles model estimates baseline probabilities before any news is considered.
2. **Qualitative research** — a ReAct agent gathers injuries, suspensions, lineup news, and context from web search and the local vector store.
3. **Synthesis** — a bounded LLM decision recommends how strongly the baseline should move and whether the opportunity is even worth considering.
4. **Guardrails** — deterministic quality checks decide whether the signal is safe enough to alert on.

This separation is important for the course because it shows that the project is not “just one prompt.” It is a structured AI system with clear responsibilities, typed state, persistence, and explicit safety logic.


### The Four Agents In Plain English

Although the forecasting pipeline is often described as *three core agents*, the implementation uses **four collaborating agent classes**:

| Agent | Role in the workflow | Why it exists |
|---|---|---|
| `SupervisorAgent` | Owns the LangGraph state machine and routes control between nodes | Separates orchestration from business logic |
| `StatsMarketAgent` | Fetches fixtures/odds and computes the statistical baseline | Creates the quantitative prior |
| `NewsContextAgent` | Runs the ReAct research loop with tools and RAG | Adds live, qualitative context |
| `SynthesisAlertAgent` | Combines baseline, evidence, and odds into a final decision | Produces the adjusted forecast and alert outcome |

In other words:

- `StatsMarketAgent` answers: **What do the numbers say before news?**
- `NewsContextAgent` answers: **What important context might the baseline be missing?**
- `SynthesisAlertAgent` answers: **Given both, is there a real edge worth alerting on?**
- `SupervisorAgent` answers: **What runs next, and when do we stop?**


### LangGraph And Shared State

The project uses **LangGraph** because the workflow is not a single straight line. Each upcoming fixture moves through a controlled state machine:

`stats_market → setup_next_match → news_context → synthesis → next match`

LangGraph is a good fit here because it provides:

- a **typed shared state** object (`GraphState`)
- explicit **node boundaries**
- conditional routing and early exit
- a clean match-loop pattern for processing many fixtures in one run

This means the notebook is demonstrating an actual multi-agent orchestration framework, not just several classes called in sequence.


In [1]:
import inspect
from soccer_forecast_agent.agents.supervisor import GraphState, SupervisorAgent

print("=== GraphState ===")
print(inspect.getsource(GraphState))
print()
print("=== SupervisorAgent.build_graph ===")
print(inspect.getsource(SupervisorAgent.build_graph))


=== GraphState ===
class GraphState(TypedDict):
    """Shared state passed between all agent nodes in the LangGraph graph."""

    competition: str
    days_ahead: int
    matches: list[Match]
    odds_map: dict[str, MarketOdds]
    baseline_forecasts: dict[str, BaselineForecast]
    pending_match_ids: list[str]
    current_match_id: str | None
    current_forecast_id: str | None
    evidence_items: list[EvidenceItem]
    interpreted_evidence: list[InterpretedEvidence]
    forecast: Forecast | None
    all_forecasts: list[Forecast]
    alert_sent: bool
    errors: list[str]


=== SupervisorAgent.build_graph ===
    def build_graph(self) -> CompiledStateGraph:
        """Construct and compile the LangGraph StateGraph with all nodes and conditional edges."""
        graph = StateGraph(GraphState)

        graph.add_node("stats_market", self._stats.run)
        graph.add_node("setup_next_match", self._setup_next_match)
        graph.add_node("news_context", self._news.run)
        graph.a

### Shared State Evolution Through The Graph

The LangGraph state is the object that connects all agents. The most important fields evolve as follows:

| Stage | Important fields added or updated |
|---|---|
| Initial state | `competition`, `days_ahead`, empty maps/lists |
| After `StatsMarketAgent` | `matches`, `odds_map`, `baseline_forecasts`, `pending_match_ids` |
| After `setup_next_match` | `current_match_id`, `current_forecast_id`, clean per-match evidence state |
| After `NewsContextAgent` | `evidence_items`, `interpreted_evidence` |
| After `SynthesisAlertAgent` | `forecast`, `alert_sent`, `all_forecasts` |

This is worth highlighting for the course because the notebook is showing a **stateful agent graph**, not a stateless prompt pipeline.


### MCP Server And Tool Boundary

The system also includes an **MCP server layer** (`mcp_server/server.py`). Conceptually, this sits between the agents and the data tools.

Why this matters:

- the agents do **not** need to know the details of football-data.org, Tavily, or The Odds API
- tools can be exposed through a standard interface and reused by other clients
- the design is easier to extend or deploy later because tool execution is separated from agent reasoning

In the local demo path, the notebook uses `LocalMCPToolClient` as a lightweight stand-in for the same boundary. In the full architecture, the exposed MCP tools are:

- `get_fixtures`
- `get_odds`
- `search_news`

So even though the notebook often runs locally for convenience, the architecture still respects the “agents call tools through a tool layer” design.


In [2]:
import inspect
from soccer_forecast_agent.mcp_server.server import create_server

print("=== MCP server tool registration ===")
print(inspect.getsource(create_server))


=== MCP server tool registration ===
def create_server(fixture_fetcher, odds_fetcher, search_tool) -> Server:
    """Build and return the MCP server with all tools registered. Dependencies are injected."""
    server = Server("soccer-forecast-tools")

    @server.list_tools()
    async def list_tools() -> list[types.Tool]:
        """Advertise available tools to MCP clients."""
        return [
            types.Tool(
                name="get_fixtures",
                description="Return upcoming Premier League fixtures within the next N days.",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "days_ahead": {"type": "integer", "default": 7},
                    },
                },
            ),
            types.Tool(
                name="get_odds",
                description="Return current market odds for a specific match.",
                inputSchema={
                    "type": "object",
       

---

## 1 · Environment Setup

In [3]:
import inspect, sqlite3, os, sys, subprocess
from datetime import datetime, timedelta, timezone
from IPython.display import HTML, display

sys.path.insert(0, os.path.abspath("."))
from dotenv import load_dotenv
load_dotenv()

from soccer_forecast_agent.config import Config
from soccer_forecast_agent.memory.sqlite_repository import SQLiteRepository, init_db
from soccer_forecast_agent.memory.seed_sources import seed_source_reliability
from soccer_forecast_agent.models.match import (
    Match, MarketOdds, BaselineForecast, Forecast, MatchContext, AlertPayload,
)
from soccer_forecast_agent.models.evidence import EvidenceItem, InterpretedEvidence

config = Config.from_env()
conn   = sqlite3.connect(config.db_path)
conn.row_factory = sqlite3.Row
init_db(conn)
seed_source_reliability(conn)
repo = SQLiteRepository(conn)

resolved = repo.get_all_finished("PL")
print(f"Config loaded — LLM provider : {config.llm_provider}  model : {config.llm_model}")
print(f"Embedding provider : {config.embedding_provider}")
print(f"SQLite path        : {config.db_path}")
print(f"Resolved matches in DB : {len(resolved)}")


Config loaded — LLM provider : openai  model : gpt-4.1-mini
Embedding provider : huggingface
SQLite path        : soccer_forecast.db
Resolved matches in DB : 385


---

## 2 · Core Data Models (Phase 1)

All structured data flows as typed dataclasses — no raw `dict` in any public interface.

In [4]:
from soccer_forecast_agent.models.match import Match, MarketOdds, BaselineForecast

for cls in (Match, MarketOdds, BaselineForecast):
    print(f"=== {cls.__name__} ===")
    print(inspect.getsource(cls))
    print()


=== Match ===
@dataclass
class Match:
    """A Premier League fixture with its current status and optional final score."""

    match_id: str
    competition: str
    home_team: str
    away_team: str
    kickoff_time: datetime
    status: str  # "upcoming" | "live" | "resolved"
    final_score: str | None = None


=== MarketOdds ===
@dataclass
class MarketOdds:
    """Decimal odds for a match across winner and over/under markets at a point in time."""

    odds_id: str
    match_id: str
    timestamp: datetime
    home_win: float
    draw: float
    away_win: float
    over_2_5: float
    under_2_5: float
    winner_market_source: str = "Unknown source"
    goals_market_source: str = "Unknown source"
    market_sources_seen: list[str] = field(default_factory=list)


=== BaselineForecast ===
@dataclass
class BaselineForecast:
    """Probabilities produced by the statistical baseline, independent of market odds."""

    home_win: float
    draw: float
    away_win: float
    over_2_5: f

### Repository and LLM protocols

In [5]:
from soccer_forecast_agent.providers.llm import LLMProvider
from soccer_forecast_agent.memory.repository import MatchRepository, VectorRepository

for proto in (LLMProvider, MatchRepository, VectorRepository):
    print(f"=== {proto.__name__} ===")
    print(inspect.getsource(proto))
    print()


=== LLMProvider ===
class LLMProvider(Protocol):
    """Abstract interface for language model calls. Agents depend on this, never on a concrete SDK."""

    def chat(self, messages: list[dict[str, str]], **kwargs: Any) -> str:
        """Send a conversation and return the assistant reply as a string."""
        ...

    def chat_with_tools(
        self,
        messages: list[dict[str, str]],
        tools: list[dict],
        **kwargs: Any,
    ) -> dict:
        """Send a conversation with tool definitions; return the raw response dict including any tool calls."""
        ...

    def format_assistant_turn(self, response: dict) -> dict:
        """Convert a raw chat_with_tools response into a message dict suitable for appending to history."""
        ...

    def format_tool_result(self, tool_call_id: str, content: str) -> dict:
        """Build a tool-result message for appending to history after a tool was called."""
        ...


=== MatchRepository ===
class MatchRepository(Prot

### Why Protocols Matter In This Project

The course specification emphasises modularity and SOLID design. This project uses `Protocol` interfaces so that:

- the agents do not depend on concrete OpenAI / Anthropic SDK classes
- the storage layer can swap implementations without changing agent logic
- the vector store, alert channel, and baseline strategy can all be replaced independently

This is a concrete example of the **Dependency Inversion Principle** in an applied AI system.


---

## 3 · Alert Guardrails (Phase 1)

Six hard checks must pass before any alert fires.

In [6]:
from soccer_forecast_agent.guardrails.alert_guard import AlertGuard, AlertGuardConfig

print(inspect.getsource(AlertGuardConfig))
print()
cfg = AlertGuardConfig()
print("Default thresholds:")
for field in cfg.__dataclass_fields__:
    print(f"  {field:<28}: {getattr(cfg, field)}")


@dataclass(frozen=True)
class AlertGuardConfig:
    """Tunable thresholds that define the alert quality bar."""

    min_evidence_count: int = 3
    min_avg_reliability: float = 0.5
    max_evidence_age_hours: int = 48
    min_unique_sources: int = 2
    min_edge_threshold: float = 0.05
    min_confidence_threshold: float = 0.60
    spam_window_hours: int = 6
    min_odds_delta: float = 0.05


Default thresholds:
  min_evidence_count          : 3
  min_avg_reliability         : 0.5
  max_evidence_age_hours      : 48
  min_unique_sources          : 2
  min_edge_threshold          : 0.05
  min_confidence_threshold    : 0.6
  spam_window_hours           : 6
  min_odds_delta              : 0.05


---

## 4 · Statistical Baseline — Dixon-Coles Model (Phases 2 & 7)

Two baseline strategies are implemented and compared:

| Strategy | Winner Brier ↓ | Goals Brier ↓ | Notes |
|---|---|---|---|
| `SimpleBaselineStrategy` | 0.2207 | 0.2635 | Rolling 5-game form |
| `EnhancedBaselineStrategy` | 0.2202 | **0.2544** | Recency-weighted form, blended xG |
| `DixonColesStrategy` | **0.2198** | 0.2673 | MLE on full season, τ correction |

Random baseline ≈ 0.222 (winner), 0.250 (goals). All three strategies beat random on the winner market.
**Dixon-Coles is used as the default** — it wins on the primary market and provides per-team attack/defence parameters.


### What `StatsMarketAgent` Does

`StatsMarketAgent` is the **quantitative entry point** of the pipeline.

For each upcoming fixture it:

1. fetches the match metadata
2. fetches the current market odds
3. extracts recent team history
4. builds a `MatchContext`
5. computes a baseline forecast using the injected strategy

This agent deliberately does **not** call the LLM. Its job is to create a disciplined statistical prior before any qualitative reasoning happens.


In [7]:
import inspect
from soccer_forecast_agent.agents.stats_market import StatsMarketAgent

print("=== StatsMarketAgent.run ===")
print(inspect.getsource(StatsMarketAgent.run))


=== StatsMarketAgent.run ===
    def run(self, state: GraphState) -> GraphState:
        """Fetch fixtures and odds, compute baseline forecasts, and update shared state."""
        competition = state["competition"]
        days_ahead = state["days_ahead"]
        matches = self._fixtures.fetch_upcoming(competition=competition, days_ahead=days_ahead)

        odds_map = dict(state.get("odds_map", {}))
        baseline_forecasts = dict(state.get("baseline_forecasts", {}))
        errors = list(state.get("errors", []))

        for match in matches:
            self._match_repo.save_match(match)
            odds = self._odds.fetch_odds(match.home_team, match.away_team)
            if odds is None:
                errors.append(f"No odds found for {match.home_team} vs {match.away_team}")
                continue

            home_history = self._match_repo.get_recent_finished(match.home_team, competition=competition, limit=5)
            away_history = self._match_repo.get_recent_finished

In [8]:
from soccer_forecast_agent.analytics.dixon_coles import DixonColesStrategy
from soccer_forecast_agent.analytics.baseline import EnhancedBaselineStrategy
from soccer_forecast_agent.analytics.features import FeatureExtractor

# Fit Dixon-Coles on all resolved PL matches in the database
dc = DixonColesStrategy()
dc.fit(resolved)

print(f"Dixon-Coles fitted on {len(resolved)} matches")
p = dc._params
print(f"  Teams modelled : {len(p.teams)}")
print(f"  Home advantage : {p.home_adv:.3f}  (log scale)")
print(f"  ρ (rho)        : {p.rho:.3f}  (low-score correlation)")
print()

# Show attack / defence parameters for selected clubs
print(f"{'Team':<32} {'Attack':>8} {'Defence':>8}")
print("-" * 50)
top_teams = ["Arsenal", "Liverpool", "Manchester City",
             "Nottingham Forest", "Chelsea", "Burnley"]
for team in top_teams:
    if team in p.attack:
        att = p.attack[team]
        dfc = p.defense[team]
        print(f"  {team:<30} {att:>+8.3f} {dfc:>+8.3f}")


Dixon-Coles fitted on 385 matches
  Teams modelled : 23
  Home advantage : 0.175  (log scale)
  ρ (rho)        : -0.137  (low-score correlation)

Team                               Attack  Defence
--------------------------------------------------
  Arsenal                          +0.000   +0.063
  Liverpool                        -0.072   +0.535
  Manchester City                  +0.003   +0.052
  Nottingham Forest                -0.423   +0.498
  Chelsea                          -0.234   +0.480
  Burnley                          -0.621   +0.911


### Live baseline prediction for two upcoming fixtures

In [9]:
from soccer_forecast_agent.domain.team_names import TeamNameNormalizer

norm = TeamNameNormalizer()
extractor = FeatureExtractor()

placeholder_odds = MarketOdds(
    odds_id="demo", match_id="demo",
    timestamp=datetime.now(timezone.utc),
    home_win=2.1, draw=3.4, away_win=3.6,
    over_2_5=1.9, under_2_5=1.9,
)

pairs = [("Bournemouth", "Manchester City"), ("Aston Villa", "Liverpool")]
for home_raw, away_raw in pairs:
    home = norm.canonicalize(home_raw)
    away = norm.canonicalize(away_raw)

    home_hist = repo.get_recent_finished(home, "PL", limit=5)
    away_hist = repo.get_recent_finished(away, "PL", limit=5)

    def results(team, hist):
        out = []
        for m in hist:
            if m.final_score:
                h, a = map(int, m.final_score.split("-"))
                if m.home_team == team:
                    out.append("W" if h > a else ("D" if h == a else "L"))
                else:
                    out.append("W" if a > h else ("D" if h == a else "L"))
        return out

    def goals(team, hist, scored=True):
        vals = []
        for m in hist:
            if m.final_score:
                h, a = map(int, m.final_score.split("-"))
                g = h if m.home_team == team else a
                if not scored:
                    g = a if m.home_team == team else h
                vals.append(float(g))
        return vals

    ctx = extractor.extract(
        match=Match(match_id=f"{home}-{away}", competition="PL",
                    home_team=home, away_team=away,
                    kickoff_time=datetime.now(timezone.utc) + timedelta(days=1),
                    status="upcoming"),
        odds=placeholder_odds,
        home_results=results(home, home_hist),
        away_results=results(away, away_hist),
        home_goals_scored=goals(home, home_hist, True),
        home_goals_conceded=goals(home, home_hist, False),
        away_goals_scored=goals(away, away_hist, True),
        away_goals_conceded=goals(away, away_hist, False),
    )

    bl_dc  = dc.compute(ctx)
    bl_enh = EnhancedBaselineStrategy().compute(ctx)

    print(f"\n{'='*56}")
    print(f"  {home}  vs  {away}")
    print(f"{'='*56}")
    print(f"  {'Outcome':<14} {'Dixon-Coles':>12} {'Enhanced':>10}")
    print(f"  {'-'*14} {'-'*12} {'-'*10}")
    for label, dc_p, en_p in [
        ("Home win",    bl_dc.home_win,  bl_enh.home_win),
        ("Draw",        bl_dc.draw,      bl_enh.draw),
        ("Away win",    bl_dc.away_win,  bl_enh.away_win),
        ("Over 2.5",    bl_dc.over_2_5,  bl_enh.over_2_5),
        ("Under 2.5",   bl_dc.under_2_5, bl_enh.under_2_5),
    ]:
        print(f"  {label:<14} {dc_p:>11.1%} {en_p:>9.1%}")



  Bournemouth  vs  Manchester City
  Outcome         Dixon-Coles   Enhanced
  -------------- ------------ ----------
  Home win             18.7%     31.0%
  Draw                 25.3%     26.0%
  Away win             56.0%     43.0%
  Over 2.5             55.0%     55.3%
  Under 2.5            45.0%     44.7%

  Aston Villa  vs  Liverpool
  Outcome         Dixon-Coles   Enhanced
  -------------- ------------ ----------
  Home win             37.2%     29.0%
  Draw                 28.1%     26.0%
  Away win             34.7%     45.0%
  Over 2.5             54.7%     59.6%
  Under 2.5            45.3%     40.4%


---

## 5 · News Context Agent — Dual Retrieval (Phase 3)

The `NewsContextAgent` runs a **ReAct (Reason + Act)** loop:

1. **Seed** — query ChromaDB for pre-indexed articles relevant to the match
2. **Think** — LLM reasons about what to search next
3. **Act** — dispatch `search_news` tool call to Tavily
4. **Observe** — extract structured `EvidenceItem` objects from results
5. **Stop** — when evidence count and reliability thresholds are met


### Why RAG Is Needed Here

The project uses a **Retrieval-Augmented Generation (RAG)** design because soccer news is both:

- **time-sensitive** — injuries and lineup news can change on the same day
- **context-dependent** — some useful information comes from accumulated prior articles, not only the latest search result

That is why `NewsContextAgent` uses **dual retrieval**:

1. **Vector retrieval from ChromaDB** to recover previously indexed article chunks about the teams
2. **Live web search** to capture breaking news that is not yet in the vector store

This is an important design decision:

- if we used only live search, the system would forget useful historical context
- if we used only RAG, the system would miss same-day updates

The vector store therefore acts as long-term memory, while Tavily search acts as short-term perception.


### What `NewsContextAgent` Actually Produces

The output of the research loop is **not** a final prediction. Instead, it produces structured evidence:

- source
- summary
- direction (`home_positive`, `away_positive`, `neutral`, `uncertainty`)
- reliability score
- market scope (`winner`, `goals`, or `both`)

This evidence becomes the input to the synthesis stage. That separation is pedagogically important: the notebook is showing a pipeline where unstructured text is converted into structured reasoning artifacts before any forecast is adjusted.


In [10]:
import inspect
from soccer_forecast_agent.agents.news_context import NewsContextAgent, ToolDispatcher

print("=== ToolDispatcher ===")
print(inspect.getsource(ToolDispatcher))
print()
print("=== NewsContextAgent.run ===")
print(inspect.getsource(NewsContextAgent.run))


=== ToolDispatcher ===
class ToolDispatcher:
    """Routes LLM tool call requests to the appropriate MCP client method."""

    SUPPORTED_TOOLS = {"search_news", "get_fixtures", "get_odds"}

    def __init__(self, mcp_client) -> None:
        """Initialise with an injected MCP client."""
        self._mcp = mcp_client

    def dispatch(self, tool_name: str, arguments: dict) -> str:
        """Call the named MCP tool and return the result as a string."""
        if self._mcp is None:
            raise ValueError("Tool dispatcher requires an MCP client.")
        if tool_name not in self.SUPPORTED_TOOLS:
            raise ValueError(f"Unknown tool: {tool_name}")
        if hasattr(self._mcp, "call_tool"):
            return self._serialise(self._mcp.call_tool(tool_name, arguments))
        if hasattr(self._mcp, tool_name):
            return self._serialise(getattr(self._mcp, tool_name)(**arguments))
        raise ValueError(f"Unknown tool: {tool_name}")

    def _serialise(self, value) 

In [11]:
import chromadb

chroma_client = chromadb.PersistentClient(path=config.chroma_path)

try:
    from soccer_forecast_agent.memory.chroma_repository import ChromaVectorRepository
    from soccer_forecast_agent.providers.embeddings import SentenceTransformerEmbeddingProvider

    embedding_provider = SentenceTransformerEmbeddingProvider(model_name=config.embedding_model)
    vector_repo = ChromaVectorRepository(client=chroma_client, embedding_provider=embedding_provider)

    query = "Bournemouth Manchester City injuries suspensions form"
    chunks = vector_repo.search(query=query, teams=["Bournemouth", "Manchester City"], top_k=4)

    print(f"Vector store query : '{query}'")
    print(f"Chunks retrieved   : {len(chunks)}")
    print()
    for i, chunk in enumerate(chunks, 1):
        print(f"  [{i}] {chunk.source}")
        print(f"       {chunk.content[:140].strip()}...")
        print()
except Exception as exc:
    print("Semantic retrieval demo skipped:", exc)
    print("Falling back to a direct Chroma collection preview so the notebook remains runnable.")
    collection = chroma_client.get_or_create_collection("soccer_news")
    preview = collection.peek(limit=4)
    documents = preview.get("documents", [])
    metadatas = preview.get("metadatas", [])
    print(f"Stored chunks available: {collection.count()}")
    print()
    for i, doc in enumerate(documents, 1):
        meta = metadatas[i - 1] if i - 1 < len(metadatas) else {}
        source = meta.get("source", "unknown source")
        print(f"  [{i}] {source}")
        print(f"       {str(doc)[:140].strip()}...")
        print()


/Users/lizethbuendia/Desktop/SoccerForecastAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|                                | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 8495.67it/s]

Vector store query : 'Bournemouth Manchester City injuries suspensions form'
Chunks retrieved   : 4

  [1] https://www.transfermarkt.us/bournemouth-afc/sperrenundverletzungen/verein/989
       AFC Bournemouth - Suspensions and injuries | Transfermarkt # AFC Bournemouth. AFC Bournemouth U21AFC Bournemouth U21. AFC Bournemouth Jugend...

  [2] https://www.newsnow.com/us/Sports/Soccer/Premier+League/Manchester+City/Injuries+and+Suspensions
       Manchester City Injuries and Suspensions News - NewsNow Latest news on Manchester City injuries and suspensions, covering squad updates, fit...

  [3] https://www.premierinjuries.com/teams/afc-bournemouth
       AFC Bournemouth - Premier Injuries Which AFC Bournemouth players are injured? Details of all the current injuries, suspensions and absences,...

  [4] https://www.newsnow.com/us/Sports/Soccer/Premier+League/Bournemouth/Injuries+and+Suspensions
       AFC Bournemouth Injuries & Suspensions news - NewsNow Dutch forward Justin Kluivert retur

### EvidenceItem — structured output of the research loop

In [12]:
import inspect
from soccer_forecast_agent.models.evidence import EvidenceItem

print(inspect.getsource(EvidenceItem))


@dataclass
class EvidenceItem:
    """A single piece of qualitative evidence gathered by the ReAct research agent."""

    evidence_id: str
    forecast_id: str
    source: str
    url: str
    timestamp: datetime
    summary: str
    direction: str        # "home_positive" | "away_positive" | "neutral" | "uncertainty"
    reliability_score: float  # 0.0 – 1.0
    applies_to_market: str    # "winner" | "goals" | "both"



---

## 6 · Synthesis & Alert Agent (Phase 4)

`SynthesisAlertAgent` is the most complex agent in the system. It orchestrates five sub-steps:

1. **Evidence quality scoring** — deterministic score from reliability, source diversity, and count
2. **LLM synthesis decision** — structured JSON output with bounded adjustment labels and conviction score
3. **Bounded log-odds adjustment** — moves baseline probabilities in log-odds space, scaled by `evidence_quality × llm_conviction`
4. **Edge calculation** — adjusted probability minus overround-normalised market-implied probability
5. **Dual guard** — `AlertGuard` hard checks AND LLM `alert_worthy` flag must both pass before an alert is sent

All tunable parameters are consolidated in `SynthesisTuning` and injected at construction time.


### What `SynthesisAlertAgent` Contributes

This is the agent that turns research into an actionable decision.

Its job is **not** to freehand probabilities from scratch. Instead, it:

- receives the baseline from `StatsMarketAgent`
- receives structured evidence from `NewsContextAgent`
- asks the LLM for a bounded synthesis decision
- translates that decision into code-controlled probability movement
- compares the adjusted forecast against market-implied probabilities
- applies hard alert guardrails

This design is important academically because it keeps the LLM influential but bounded. The model contributes judgment, while the code keeps control over normalization, caps, edge math, persistence, and spam suppression.


In [13]:
import inspect
from soccer_forecast_agent.agents.synthesis_alert import SynthesisAlertAgent

print("=== SynthesisAlertAgent.run ===")
print(inspect.getsource(SynthesisAlertAgent.run))


=== SynthesisAlertAgent.run ===
    def run(self, state: GraphState) -> GraphState:
        """Apply LLM-guided bounded adjustment, evaluate the guard, send alert if thresholds pass, and persist forecast."""
        current_match_id = state.get("current_match_id")
        current_forecast_id = state.get("current_forecast_id")
        errors = list(state.get("errors", []))
        if not current_match_id:
            errors.append("SynthesisAlertAgent requires current_match_id in state")
            return {**state, "errors": errors}
        if not current_forecast_id:
            errors.append("SynthesisAlertAgent requires current_forecast_id in state")
            return {**state, "errors": errors}

        match = self._match_from_state(state, current_match_id)
        if match is None:
            errors.append(f"Match not found for current_match_id={current_match_id}")
            return {**state, "errors": errors}

        baseline = state.get("baseline_forecasts", {}).get(current

In [14]:
from soccer_forecast_agent.agents.synthesis_alert import SynthesisAlertAgent, SynthesisTuning
from soccer_forecast_agent.models.match import SynthesisDecision, SynthesisResult

print("=== SynthesisTuning — all tuning knobs in one injectable dataclass ===")
print(inspect.getsource(SynthesisTuning))


=== SynthesisTuning — all tuning knobs in one injectable dataclass ===
@dataclass(frozen=True)
class SynthesisTuning:
    """Typed tuning values for bounded synthesis adjustments and confidence scoring."""

    base_sensitivity: float = 0.15
    probability_floor: float = 0.01
    probability_ceiling: float = 0.99
    implied_probability_default: float = 0.0
    prior_odds_delta_default: float = 0.0
    evidence_quality_cap: float = 1.0
    evidence_quality_empty: float = 0.0
    source_bonus_per_unique_source: float = 0.05
    source_bonus_cap: float = 0.15
    count_bonus_per_item: float = 0.03
    count_bonus_cap: float = 0.15
    confidence_base: float = 0.15
    confidence_quality_weight: float = 0.45
    confidence_llm_weight: float = 0.35
    confidence_cap: float = 0.95
    rounding_precision: int = 4
    decision_evidence_limit: int = 6
    decision_temperature: float = 0.0
    decision_max_tokens: int = 350



In [15]:
print("=== SynthesisDecision — structured JSON output from the LLM ===")
print(inspect.getsource(SynthesisDecision))


=== SynthesisDecision — structured JSON output from the LLM ===
@dataclass
class SynthesisDecision:
    """Structured LLM adjudication over how the baseline should move and whether the signal is alert-worthy."""

    market_category: str
    recommended_market: str
    winner_adjustment: str
    goals_adjustment: str
    llm_conviction_score: float
    alert_worthy: bool
    rationale_points: list[str] = field(default_factory=list)
    risk_points: list[str] = field(default_factory=list)
    summary: str = ""



In [16]:
print("=== Bounded adjustment lookup tables ===")
print("WINNER_ADJUSTMENTS (home_delta, away_delta in log-odds units):")
for k, v in SynthesisAlertAgent.WINNER_ADJUSTMENTS.items():
    print(f"  {k:<18} home {v[0]:+.1f}  away {v[1]:+.1f}")
print()
print("GOALS_ADJUSTMENTS (over_delta, under_delta in log-odds units):")
for k, v in SynthesisAlertAgent.GOALS_ADJUSTMENTS.items():
    print(f"  {k:<18} over {v[0]:+.1f}  under {v[1]:+.1f}")


=== Bounded adjustment lookup tables ===
WINNER_ADJUSTMENTS (home_delta, away_delta in log-odds units):
  strong_home        home +1.0  away -1.0
  medium_home        home +0.7  away -0.7
  light_home         home +0.3  away -0.3
  neutral            home +0.0  away +0.0
  light_draw         home -0.2  away -0.2
  medium_draw        home -0.4  away -0.4
  light_away         home -0.3  away +0.3
  medium_away        home -0.7  away +0.7
  strong_away        home -1.0  away +1.0

GOALS_ADJUSTMENTS (over_delta, under_delta in log-odds units):
  strong_over        over +1.0  under -1.0
  medium_over        over +0.7  under -0.7
  light_over         over +0.3  under -0.3
  neutral            over +0.0  under +0.0
  light_under        over -0.3  under +0.3
  medium_under       over -0.7  under +0.7
  strong_under       over -1.0  under +1.0


### Evidence quality scoring and confidence formula

In [17]:
print(inspect.getsource(SynthesisAlertAgent._evidence_quality_score))
print()
print(inspect.getsource(SynthesisAlertAgent._confidence_score))


    def _evidence_quality_score(self, interpreted: list[InterpretedEvidence]) -> float:
        """Estimate deterministic evidence quality from reliability, source diversity, and count."""
        if not interpreted:
            return self._tuning.evidence_quality_empty
        avg_reliability = sum(item.reliability_score for item in interpreted) / len(interpreted)
        source_bonus = min(
            len({item.source for item in interpreted}) * self._tuning.source_bonus_per_unique_source,
            self._tuning.source_bonus_cap,
        )
        count_bonus = min(
            len(interpreted) * self._tuning.count_bonus_per_item,
            self._tuning.count_bonus_cap,
        )
        score = min(self._tuning.evidence_quality_cap, avg_reliability + source_bonus + count_bonus)
        return round(score, self._tuning.rounding_precision)


    def _confidence_score(self, evidence_quality_score: float, llm_conviction_score: float) -> float:
        """Blend deterministic eviden

### Synthesis walkthrough — Bournemouth vs Manchester City

In [18]:
import math

# Reproduce the Bournemouth vs Man City alert using the real agent internals
# ── inputs ────────────────────────────────────────────────────────────────
baseline_home  = 0.385   # Dixon-Coles home-win
baseline_draw  = 0.327
baseline_away  = 0.288
market_home    = 0.221   # implied from William Hill odds

# ── evidence quality ──────────────────────────────────────────────────────
# 3 evidence items from BBC, Sky Sports, Transfermarkt (reliability ~0.75 avg)
avg_reliability   = 0.75
source_bonus      = min(3 * 0.05, 0.15)   # 3 unique sources
count_bonus       = min(3 * 0.03, 0.15)   # 3 items
evidence_quality  = min(1.0, avg_reliability + source_bonus + count_bonus)

# ── LLM synthesis decision (returned JSON) ────────────────────────────────
llm_conviction    = 0.70
winner_adjustment = "medium_home"          # LLM chose medium_home
goals_adjustment  = "neutral"

# ── two-factor scale ──────────────────────────────────────────────────────
scale = evidence_quality * llm_conviction

# ── log-odds adjustment ───────────────────────────────────────────────────
base_sensitivity = 0.15
home_delta, away_delta = SynthesisAlertAgent.WINNER_ADJUSTMENTS[winner_adjustment]

def log_odds_adjust(p, delta, sensitivity=0.15):
    p = min(max(p, 0.01), 0.99)
    return 1 / (1 + math.exp(-(math.log(p / (1 - p)) + delta * scale * sensitivity)))

raw_home  = log_odds_adjust(baseline_home, home_delta)
raw_away  = log_odds_adjust(baseline_away, away_delta)
raw_draw  = baseline_draw
total     = raw_home + raw_draw + raw_away
adj_home  = raw_home / total
adj_draw  = raw_draw / total
adj_away  = raw_away / total

# ── edge and confidence ───────────────────────────────────────────────────
tuning    = SynthesisTuning()
edge      = adj_home - market_home
confidence = min(
    tuning.confidence_cap,
    tuning.confidence_base
    + evidence_quality * tuning.confidence_quality_weight
    + llm_conviction   * tuning.confidence_llm_weight,
)

print("Synthesis walkthrough — Bournemouth vs Manchester City")
print()
print(f"  Evidence quality score        : {evidence_quality:.2f}")
print(f"    avg reliability             : {avg_reliability:.2f}")
print(f"    source diversity bonus      : +{source_bonus:.2f}")
print(f"    count bonus                 : +{count_bonus:.2f}")
print()
print(f"  LLM decision")
print(f"    winner_adjustment           : {winner_adjustment}")
print(f"    goals_adjustment            : {goals_adjustment}")
print(f"    llm_conviction_score        : {llm_conviction:.2f}")
print()
print(f"  Two-factor scale              : {evidence_quality:.2f} × {llm_conviction:.2f} = {scale:.3f}")
print()
print(f"  Probabilities")
print(f"    {'Outcome':<18} {'Baseline':>10} {'Adjusted':>10}")
print(f"    {'-'*18} {'-'*10} {'-'*10}")
print(f"    {'Home win':<18} {baseline_home:>9.1%} {adj_home:>9.1%}")
print(f"    {'Draw':<18} {baseline_draw:>9.1%} {adj_draw:>9.1%}")
print(f"    {'Away win':<18} {baseline_away:>9.1%} {adj_away:>9.1%}")
print()
print(f"  Edge = {adj_home:.1%} − {market_home:.1%} = {edge:+.1%}")
print(f"  Confidence score              : {confidence:.1%}")
print()
if edge >= tuning.min_edge_threshold if hasattr(tuning, 'min_edge_threshold') else 0.05:
    pass
if edge >= 0.05 and confidence >= 0.60:
    print("  AlertGuard: edge ✅  confidence ✅  → alert_worthy check next")
    print("  LLM alert_worthy=True → ALERT SENT ✅")
else:
    print("  AlertGuard: FAILED → alert withheld")


Synthesis walkthrough — Bournemouth vs Manchester City

  Evidence quality score        : 0.99
    avg reliability             : 0.75
    source diversity bonus      : +0.15
    count bonus                 : +0.09

  LLM decision
    winner_adjustment           : medium_home
    goals_adjustment            : neutral
    llm_conviction_score        : 0.70

  Two-factor scale              : 0.99 × 0.70 = 0.693

  Probabilities
    Outcome              Baseline   Adjusted
    ------------------ ---------- ----------
    Home win               38.5%     40.1%
    Draw                   32.7%     32.6%
    Away win               28.8%     27.3%

  Edge = 40.1% − 22.1% = +18.0%
  Confidence score              : 84.0%

  AlertGuard: edge ✅  confidence ✅  → alert_worthy check next
  LLM alert_worthy=True → ALERT SENT ✅


---

## 7 · Full End-to-End Pipeline Run  `[LIVE — requires API keys]`

The `SupervisorAgent` orchestrates the complete match-loop:
for each upcoming fixture → StatsMarketAgent → (if edge) NewsContextAgent → SynthesisAlertAgent


In [19]:
# [LIVE] Full pipeline run — optional
# This cell no longer clears the database automatically.
# That makes the notebook safer to rerun and allows graceful fallback when live APIs fail.

import sqlite3 as _sqlite3
live_run_ok = False
forecast_ids = []
print("Live pipeline cell prepared. Existing forecasts are preserved unless a successful live run replaces them.\n")


Live pipeline cell prepared. Existing forecasts are preserved unless a successful live run replaces them.



In [20]:
# [LIVE] Run the supervisor
try:
    import openai, chromadb as _chromadb
    from soccer_forecast_agent.providers.openai_provider import OpenAIProvider
    from soccer_forecast_agent.memory.chroma_repository import ChromaVectorRepository
    from soccer_forecast_agent.providers.embeddings import SentenceTransformerEmbeddingProvider
    from soccer_forecast_agent.tools.fixtures import FixtureFetcher
    from soccer_forecast_agent.tools.odds import OddsFetcher
    from soccer_forecast_agent.tools.search import WebSearchTool
    from soccer_forecast_agent.tools.email_sender import ConsoleAlertChannel
    from soccer_forecast_agent.analytics.dixon_coles import DixonColesStrategy
    from soccer_forecast_agent.analytics.features import FeatureExtractor
    from soccer_forecast_agent.guardrails.alert_guard import AlertGuard, AlertGuardConfig
    from soccer_forecast_agent.agents.stats_market import StatsMarketAgent
    from soccer_forecast_agent.agents.news_context import NewsContextAgent, ToolDispatcher
    from soccer_forecast_agent.agents.synthesis_alert import SynthesisAlertAgent, SynthesisTuning
    from soccer_forecast_agent.agents.supervisor import SupervisorAgent
    from soccer_forecast_agent.main import LocalMCPToolClient

    _llm = OpenAIProvider(openai.OpenAI(api_key=config.openai_api_key), model=config.llm_model)
    _chroma = _chromadb.PersistentClient(path=config.chroma_path)
    _embed  = SentenceTransformerEmbeddingProvider(model_name=config.embedding_model)
    _vrepo  = ChromaVectorRepository(client=_chroma, embedding_provider=_embed)

    _fix    = FixtureFetcher(config.football_data_api_key)
    _odds   = OddsFetcher(config.odds_api_key)
    _search = WebSearchTool(config.search_api_key)
    _mcp    = LocalMCPToolClient(_fix, _odds, _search)

    _fresh_conn = _sqlite3.connect(config.db_path)
    _fresh_conn.row_factory = _sqlite3.Row
    from soccer_forecast_agent.memory.sqlite_repository import SQLiteRepository, init_db
    from soccer_forecast_agent.memory.seed_sources import seed_source_reliability
    init_db(_fresh_conn)
    seed_source_reliability(_fresh_conn)
    _repo2 = SQLiteRepository(_fresh_conn)

    _resolved2 = _repo2.get_all_finished("PL")
    _dc2 = DixonColesStrategy().fit(_resolved2) if len(_resolved2) >= 20 else None

    _stats  = StatsMarketAgent(_fix, _odds, _dc2 or DixonColesStrategy(), FeatureExtractor(), _repo2)
    _news   = NewsContextAgent(_llm, ToolDispatcher(_mcp), _vrepo, _repo2, _repo2, config.react_max_steps)
    _guard  = AlertGuard(AlertGuardConfig(
        min_edge_threshold=config.min_edge_threshold,
        min_confidence_threshold=config.min_confidence_threshold,
        spam_window_hours=config.spam_window_hours,
    ))
    _synth  = SynthesisAlertAgent(
        _llm, ConsoleAlertChannel(), _repo2, _repo2, _guard,
        tuning=SynthesisTuning(base_sensitivity=config.base_sensitivity),
    )
    _supervisor = SupervisorAgent(_stats, _news, _synth, min_edge_threshold=config.min_edge_threshold)

    print("Running supervisor — processing all upcoming PL fixtures...\n")
    forecasts = _supervisor.run(competition="PL", days_ahead=7)
    forecast_ids = [f.forecast_id for f in forecasts]
    live_run_ok = True
    print(f"\nDone. {len(forecasts)} forecasts generated.")
except Exception as exc:
    _exc_name = exc.__class__.__name__
    _message = str(exc)
    if "401" in _message or "Unauthorized" in _message:
        _summary = "external odds API request was rejected, likely because the quota was exhausted or the key was unavailable"
    elif "429" in _message:
        _summary = "an external API rate limit was reached during the live run"
    else:
        _summary = "a live dependency failed during execution"
    print(f"Live pipeline run skipped or failed ({_exc_name}): {_summary}.")
    print("Falling back to the most recent forecasts already stored in SQLite so the notebook remains runnable.")
    _fallback_conn = _sqlite3.connect(config.db_path)
    _fallback_conn.row_factory = _sqlite3.Row
    _rows = _fallback_conn.execute(
        'SELECT forecast_id FROM forecasts ORDER BY run_timestamp DESC LIMIT 12'
    ).fetchall()
    forecast_ids = [row['forecast_id'] for row in _rows]
    _fallback_conn.close()
    print(f"Recovered {len(forecast_ids)} stored forecast ids for downstream summary cells.")


Loading weights:   0%|                                | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 9306.82it/s]

Running supervisor — processing all upcoming PL fixtures...



Live pipeline run skipped or failed (HTTPStatusError): external odds API request was rejected, likely because the quota was exhausted or the key was unavailable.
Falling back to the most recent forecasts already stored in SQLite so the notebook remains runnable.
Recovered 12 stored forecast ids for downstream summary cells.


### Pipeline results summary

In [21]:
_fresh_conn2 = _sqlite3.connect(config.db_path)
_fresh_conn2.row_factory = _sqlite3.Row

def _market_label(market, home_team, away_team):
    labels = {
        'home_win': f'{home_team} win',
        'draw': 'Draw',
        'away_win': f'{away_team} win',
        'over_2_5': 'Over 2.5 goals',
        'under_2_5': 'Under 2.5 goals',
    }
    return labels.get(market or '', market or 'n/a')

if not globals().get('forecast_ids'):
    print('No forecast ids available from a live run; attempting to show the latest stored forecasts instead.')
    _rows = _fresh_conn2.execute('SELECT forecast_id FROM forecasts ORDER BY run_timestamp DESC LIMIT 12').fetchall()
    forecast_ids = [row['forecast_id'] for row in _rows]
if not forecast_ids:
    print('No forecasts are available in SQLite yet.')
else:
    placeholders = ','.join('?' for _ in forecast_ids)
    sql = (
        'SELECT f.edge_market, f.edge_value, f.confidence_score, f.alert_sent,'
        '       f.forecast_id,'
        '       m.home_team, m.away_team, m.kickoff_time'
        ' FROM forecasts f JOIN matches m ON f.match_id = m.match_id'
        ' WHERE f.forecast_id IN (' + placeholders + ')'
        ' ORDER BY f.alert_sent DESC, f.edge_value DESC'
    )
    rows = _fresh_conn2.execute(sql, forecast_ids).fetchall()
    alerts = [r for r in rows if r['alert_sent']]
    print(f"{'Match':<42} {'Market':<16} {'Edge':>7}  {'Conf':>6}  Alert")
    print('-' * 85)
    for r in rows:
        sent  = 'YES' if r['alert_sent'] else 'no'
        edge  = f"{r['edge_value']:+.1%}" if r['edge_value'] else 'n/a'
        conf  = f"{r['confidence_score']:.0%}"
        match = f"{r['home_team']} vs {r['away_team']}"[:40]
        mkt   = _market_label(r['edge_market'], r['home_team'], r['away_team'])[:14]
        print(f'  {match:<40} {mkt:<16} {edge:>7}  {conf:>6}  {sent}')
    print()
    print(f'  Live run succeeded : {globals().get("live_run_ok", False)}')
    print(f'  Alerts sent : {len(alerts)}  /  {len(rows)} forecasts generated')
_fresh_conn2.close()


Match                                      Market              Edge    Conf  Alert
-------------------------------------------------------------------------------------
  Bournemouth vs Manchester City           Draw               +9.9%     74%  no
  Leeds vs Brighton                        Leeds win          +9.8%     65%  no
  Aston Villa vs Liverpool                 Aston Villa wi     +5.5%     72%  no
  Chelsea vs Tottenham                     Chelsea win        -5.4%     76%  no
  Newcastle vs West Ham                    Newcastle win      -5.6%     76%  no
  Chelsea vs Tottenham                     Chelsea win        -6.2%     71%  no
  Everton vs Sunderland                    Everton win       -11.6%     72%  no
  Brentford vs Crystal Palace              Brentford win     -13.8%     76%  no
  Manchester United vs Nottingham Forest   Manchester Uni    -18.3%     72%  no
  Wolves vs Fulham                         Fulham win        -19.9%     76%  no
  Bournemouth vs Manchester Cit

---

## 8 · HTML Alert Email — Rendered Inline

In [22]:
# Reconstruct the highest-edge alert and render its HTML inline

_conn3 = _sqlite3.connect(config.db_path)
_conn3.row_factory = _sqlite3.Row

if not globals().get("forecast_ids"):
    print("No forecast ids available from a live run; attempting to use the latest stored forecasts instead.")
    _rows = _conn3.execute('SELECT forecast_id FROM forecasts ORDER BY run_timestamp DESC LIMIT 12').fetchall()
    forecast_ids = [row['forecast_id'] for row in _rows]

best = None
if forecast_ids:
    _placeholders = ",".join("?" for _ in forecast_ids)
    _sql = f"""
        SELECT f.*, m.home_team, m.away_team, m.kickoff_time, m.match_id as mid,
               m.competition, m.status, m.final_score
        FROM forecasts f JOIN matches m ON f.match_id = m.match_id
        WHERE f.forecast_id IN ({_placeholders})
          AND f.alert_sent = 1
        ORDER BY f.edge_value DESC LIMIT 1
    """
    best = _conn3.execute(_sql, forecast_ids).fetchone()

if best:
    from soccer_forecast_agent.models.match import BaselineForecast, Forecast, Match, MarketOdds, AlertPayload
    from soccer_forecast_agent.tools.email_sender import EmailAlertChannel

    _match = Match(
        match_id=best['mid'],
        competition=best['competition'],
        home_team=best['home_team'],
        away_team=best['away_team'],
        kickoff_time=datetime.fromisoformat(best['kickoff_time']),
        status=best['status'],
        final_score=best['final_score'],
    )
    _bl = BaselineForecast(
        home_win=best['baseline_home_win'],
        draw=best['baseline_draw'],
        away_win=best['baseline_away_win'],
        over_2_5=best['baseline_over'],
        under_2_5=best['baseline_under'],
    )
    _forecast = Forecast(
        forecast_id=best['forecast_id'],
        match_id=best['mid'],
        run_timestamp=datetime.fromisoformat(best['run_timestamp']),
        baseline=_bl,
        adjusted_home_win=best['adjusted_home_win'],
        adjusted_draw=best['adjusted_draw'],
        adjusted_away_win=best['adjusted_away_win'],
        adjusted_over_2_5=best['adjusted_over'],
        adjusted_under_2_5=best['adjusted_under'],
        confidence_score=best['confidence_score'],
        edge_market=best['edge_market'],
        edge_value=best['edge_value'],
        alert_sent=True,
        rationale=best['rationale'] or "",
    )
    # Build a representative MarketOdds from the implied probabilities
    edge = best['edge_value'] or 0
    adj_p = best['adjusted_home_win'] or 0.4
    implied = adj_p - edge
    away_impl = 1 - implied - 0.23  # rough split
    _mkt_odds = MarketOdds(
        odds_id="demo", match_id=best['mid'],
        timestamp=datetime.now(timezone.utc),
        home_win=round(1/max(implied, 0.01), 2),
        draw=round(1/0.23, 2),
        away_win=round(1/max(away_impl, 0.01), 2),
        over_2_5=1.90,
        under_2_5=1.90,
        winner_market_source="William Hill",
        goals_market_source="William Hill",
        market_sources_seen=["William Hill", "Betfred", "Sky Bet"],
    )
    _payload = AlertPayload(
        match=_match,
        forecast=_forecast,
        market_odds=_mkt_odds,
        rationale=_forecast.rationale,
    )

    _channel = EmailAlertChannel.__new__(EmailAlertChannel)
    _html = _channel._render_html(_payload)
    display(HTML(_html))
    print(f"\nRendered alert: {_match.home_team} vs {_match.away_team}  edge={edge:+.1%}")
else:
    print("No alerts found in the available forecast set.")

_conn3.close()


No alerts found in the available forecast set.


---

## 9 · Model Validation — Backtest (Phase 7)

Chronological 70/30 train/test split. Dixon-Coles is fit on the training window only; all three
strategies are evaluated on unseen test matches.
Brier score ↓ is better. Random baseline ≈ 0.222 (winner), 0.250 (goals).


In [23]:
result = subprocess.run(
    [sys.executable, "validation/backtest.py"],
    capture_output=True, text=True, cwd=os.path.abspath(".")
)
print(result.stdout[-4000:])   # last 4000 chars — summary table
if result.returncode != 0:
    print(result.stderr[-1000:])


.2831 0.3339   0.49   0.54   0.2833 0.2904   0.20    0.40    0.2505  0.1567 
2026-04-12  Chelsea             Manchester City     0-3    A    0.22   0.60   0.1177 0.1600 ✓ 0.23   0.55   0.1190 0.1996 ✓ 0.32    0.43    0.2086  0.3227✓
2026-04-13  Manchester United   Leeds               1-2    A    0.61   0.42   0.3987 0.3339   0.60   0.44   0.3890 0.3148   0.56    0.69    0.3295  0.0992 
2026-04-18  Brentford           Fulham              0-0    D    0.41   0.56   0.2749 0.3087   0.41   0.52   0.2750 0.2752   0.51    0.62    0.2901  0.3798 
2026-04-18  Newcastle           Bournemouth         1-2    A    0.35   0.53   0.1892 0.2178 ✓ 0.33   0.51   0.1779 0.2398 ✓ 0.54    0.64    0.3243  0.1331 
2026-04-18  Leeds               Wolves              3-0    H    0.33   0.42   0.2301 0.3339   0.36   0.44   0.2051 0.3148   0.65    0.54    0.0617  0.2075✓
2026-04-18  Tottenham           Brighton            2-2    D    0.12   0.51   0.3147 0.2390   0.12   0.50   0.3158 0.2540   0.32    0.54    0.2

---

## 10 · Test Suite

Project unit tests across all system layers. Fakes replace external dependencies — no mocking of internals.

In [24]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short",
     "--ignore=tests/test_live_llm_provider.py",
     "--ignore=tests/test_live_news_context.py"],
    capture_output=True, text=True, cwd=os.path.abspath(".")
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.0.3, pluggy-1.6.0 -- /Users/lizethbuendia/Desktop/SoccerForecastAgent/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/lizethbuendia/Desktop/SoccerForecastAgent
configfile: pyproject.toml
plugins: langsmith-0.7.36, anyio-4.13.0
collecting ... collected 43 items

tests/test_alert_guard.py::test_alert_guard_passes_with_recent_diverse_reliable_evidence PASSED [  2%]
tests/test_alert_guard.py::test_alert_guard_collects_multiple_failure_reasons PASSED [  4%]
tests/test_alert_guard.py::test_alert_guard_handles_naive_datetimes_without_crashing PASSED [  6%]
tests/test_alert_guard.py::test_alert_guard_uses_default_config_values PASSED [  9%]
tests/test_baseline.py::test_simple_baseline_returns_normalized_probabilities PASSED [ 11%]
tests/test_baseline.py::test_simple_baseline_favors_stronger_home_side PASSED [ 13%]
tests/test_baseline.py::test_simple_baseline_kee

---

## 11 · Conclusion

The Soccer Forecast Agent is a fully operational multi-agent AI system built to SOLID principles:

| Capability | Implementation |
|---|---|
| Statistical forecasting | Dixon-Coles MLE model, beats random on winner Brier (0.220 vs 0.222) |
| Qualitative evidence | ReAct loop: ChromaDB seed → Tavily web search → structured EvidenceItem extraction |
| LLM adjudication | Bounded log-odds adjustment via synthesis decision JSON |
| Alert quality bar | AlertGuard: 6 hard checks (evidence count, reliability, diversity, edge, confidence, spam) |
| Styled alerts | HTML + plain-text dual-part email via SMTP / ConsoleAlertChannel fallback |
| Persistence | SQLite (fixtures, forecasts, evidence) + ChromaDB (514 article chunks, 20 teams) |
| Extensibility | LLM provider, alert channel, and baseline strategy are all swappable via env var |
| Observability | Structured logging, Brier-score backtest, weekly scheduled refresh script |

**Production roadmap (Phase 9):** xG-powered Dixon-Coles, Pinnacle line-movement signal, lineup-aware
parameter adjustment, Telegram bot alerts, rolling auto-calibration, and cloud deployment on a
$15/month VPS.

### Key Results

- The project implements a full **multi-agent architecture** with typed shared state, tool boundaries, and persistence.
- The baseline forecasting layer is not heuristic-only; it includes a fitted **Dixon-Coles statistical model**.
- The qualitative layer is not a black-box prompt; it uses **RAG + live search + structured evidence extraction**.
- The final decision layer is not freeform generation; it uses **bounded LLM synthesis under deterministic guardrails**.
- The notebook demonstrates both the **software engineering design** and the **AI reasoning pipeline** expected in a strong final course submission.

### Limitations

- Live cells depend on external APIs and can fail when quotas are exhausted.
- The goals market remains weaker than the winner market and still needs richer features such as xG, lineups, and weather.
- The qualitative layer can vary across runs because live search and LLM synthesis are probabilistic.

---
*This project is an academic decision-support assistant.
All alerts carry a conservative framing disclaimer and do not constitute financial advice.*
